# Fast Proposition Mapper for 152,111 Extracted Logic Passages
**Dự án:** LP-Prop — Large-Scale FactoidWiki Logic Audit  
**Mục tiêu:**
Tận dụng file **`factoidwiki_aligned_audit_full.csv` (91.2 MB)** đã cào và trích xuất thành công toàn bộ **152,111 đoạn văn logic** từ 41 triệu passages Wikipedia của tác giả.  
Notebook này **KHÔNG CẦN quét lại 41 triệu passage** (tiết kiệm 1 giờ chạy), mà **chỉ đọc trực tiếp các shard `proposition` tương ứng** từ `chentong00/factoid-wiki` để:
1. Ghép đúng các câu mệnh đề nguyên tử theo mã `pid` (sử dụng 25 ký tự đầu: `uid[:25]`).
2. Kiểm tra chính xác xem từ nối điều kiện/ngoại lệ có bị mô hình Propositionizer của tác giả **cắt bỏ hoàn toàn** hay không.
3. Xuất bảng kết quả thực nghiệm hoàn chỉnh và các biểu đồ xuất bản bài báo khoa học.


## 0. Cài đặt thư viện & Cấu hình môi trường (Setup)


In [ ]:
# Cài đặt huggingface_hub nếu chưa có
# !pip install -q huggingface_hub orjson pandas matplotlib seaborn tqdm

import os
import gc
import re
import json
import time
import glob
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from huggingface_hub import HfFileSystem

print("[+] Môi trường sẵn sàng cho Fast Proposition Mapping!")


## 1. Nạp file 152,111 Passages đã trích xuất (Load Extracted CSV)
Tự động tìm kiếm file `factoidwiki_aligned_audit_full.csv` trong Kaggle Input hoặc Local.


In [ ]:
# Danh sách các đường dẫn ưu tiên tìm file 91.2 MB đã trích xuất
candidate_paths = [
    # 1. Đường dẫn nếu bạn đẩy lên Dataset Kaggle:
    "/kaggle/input/**/factoidwiki_aligned_audit_full.csv",
    # 2. Đường dẫn máy local:
    r"D:\Documents\Project SEG\results\expriments\factoidwiki_audit_results\factoidwiki_aligned_audit_full.csv",
    os.path.join(os.getcwd(), "factoidwiki_aligned_audit_full.csv")
]

target_csv = None
for p in candidate_paths:
    matches = glob.glob(p, recursive=True) if "*" in p else ([p] if os.path.exists(p) else [])
    if matches:
        target_csv = matches[0]
        break

if not target_csv or not os.path.exists(target_csv):
    raise FileNotFoundError("Không tìm thấy file factoidwiki_aligned_audit_full.csv! Hãy thêm dataset chứa file này vào Kaggle.")

print(f"[+] Tìm thấy file đã trích xuất: {target_csv}")
df_passages = pd.read_csv(target_csv)
print(f"    -> Tổng số đoạn văn logic: {len(df_passages):,} passages")
print(f"    -> Số lượng shard độc nhất: {df_passages['shard'].nunique()} shards")

OUT_DIR = "/kaggle/working/factoidwiki_verified_results" if os.path.exists("/kaggle/working") else "factoidwiki_verified_results"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"[*] Thư mục xuất kết quả: {OUT_DIR}")


## 2. Cấu hình số lượng Shards cần Map
Bạn có thể chọn map một phần hoặc toàn bộ 1,000 shards:
- `MAP_MODE = "sample"`: Map 50 shards (~7,500 câu logic) $\to$ Chạy mất ~1-2 phút.
- `MAP_MODE = "large"`: Map 200 shards (~30,000 câu logic) $\to$ Chạy mất ~5-7 phút.
- `MAP_MODE = "full"`: Map toàn bộ 1,000 shards (152,111 câu logic) $\to$ Chạy mất ~15-20 phút.


In [ ]:
MAP_MODE = "full"  # Chọn: "sample" (50 shards), "large" (200 shards), hoặc "full" (toàn bộ 1000 shards)

MODE_LIMITS = {
    "sample": 50,
    "large": 200,
    "full": 1000
}

LIMIT_SHARDS = MODE_LIMITS.get(MAP_MODE, 1000)
all_shards = sorted(df_passages["shard"].unique())
shards_to_process = all_shards[:LIMIT_SHARDS]
print(f"[*] Chế độ: {MAP_MODE.upper()} | Sẽ xử lý {len(shards_to_process)} / {len(all_shards)} shards")

# Lọc các passage thuộc danh sách shards này
df_target = df_passages[df_passages["shard"].isin(shards_to_process)].copy()
print(f"[*] Tổng số đoạn văn logic sẽ đối soát: {len(df_target):,} đoạn")

# Nhóm các PID theo từng shard để tối ưu hóa việc truy vấn
pids_by_shard = df_target.groupby("shard")["pid"].apply(set).to_dict()
MAX_WORKERS = 12  # Số luồng tải song song các shard proposition


## 3. Thuật toán Stream & Ghép Proposition theo Shard (Fast Mapper Engine)


In [ ]:
fs = HfFileSystem()
REPO_PROP = "chentong00/factoid-wiki"

def parse_prop_line(d):
    """Cắt đúng 25 ký tự đầu của id để lấy Passage ID chuẩn xác"""
    uid = d.get("id", "")
    pid = uid[:25]
    c = d.get("contents", "")
    m = d.get("metadata", {})
    cs, ce = m.get("content_span", [None, None])
    if cs is not None and ce is not None:
        p_text = c[cs:ce].strip()
    elif "\n" in c:
        p_text = c.split("\n", 1)[1].strip()
    else:
        p_text = c.strip()
    return pid, p_text

def fetch_props_for_shard(shard_fname):
    """
    Chỉ stream 1 shard proposition và gom các proposition có PID nằm trong target_pids.
    """
    target_pids = pids_by_shard.get(shard_fname, set())
    if not target_pids:
        return shard_fname, {}
        
    prop_url = f"datasets/{REPO_PROP}/data/{shard_fname}"
    shard_props = defaultdict(list)
    
    try:
        with fs.open(prop_url, "rt", encoding="utf-8") as f:
            for line in f:
                try:
                    d = json.loads(line)
                    pid, p_text = parse_prop_line(d)
                    if pid in target_pids:
                        shard_props[pid].append(p_text)
                except:
                    continue
    except Exception as e:
        print(f"[!] Lỗi đọc shard {shard_fname}: {e}")
        
    return shard_fname, shard_props

print("[+] Khởi tạo thành công Fast Proposition Mapper Engine!")


## 4. Chạy Thu thập Song song Propositions (Parallel Fetch)


In [ ]:
print(f"[*] Bắt đầu tải và ánh xạ propositions cho {len(shards_to_process)} shards với {MAX_WORKERS} luồng...")
t0 = time.time()

global_props_by_pid = {}

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(fetch_props_for_shard, sf): sf for sf in shards_to_process}
    pbar = tqdm(as_completed(futures), total=len(shards_to_process), desc="Mapping Proposition Shards")
    
    for future in pbar:
        sf, s_props = future.result()
        global_props_by_pid.update(s_props)
        pbar.set_postfix({"Mapped PIDs": f"{len(global_props_by_pid):,}"})

elapsed = time.time() - t0
print(f"\n[+] HOÀN THÀNH ÁNH XẠ TRONG {elapsed/60:.2f} PHÚT!")
print(f"    -> Đã tìm thấy propositions cho {len(global_props_by_pid):,} / {len(df_target):,} đoạn văn!")


## 5. Kiểm tra Tỷ lệ Rơi rụng Điều kiện Logic Thực tế (Check Drop Rates)


In [ ]:
print("[*] Đang đối chiếu kiểm tra sự bảo tồn từ nối logic...")

def evaluate_row(row):
    pid = row["pid"]
    kw = row["keyword"]
    props = global_props_by_pid.get(pid, [])
    
    if not props:
        return 0, False, ""
        
    props_str = "\n".join([f"[{i+1}] {p}" for i, p in enumerate(props)])
    # Kiểm tra xem từ khóa logic có tồn tại trong bất kỳ proposition nào không
    preserved = any(re.search(r"\b" + re.escape(str(kw)) + r"\b", p, re.IGNORECASE) for p in props)
    return len(props), preserved, props_str

# Áp dụng kiểm tra
num_props_list = []
preserved_list = []
props_str_list = []

for idx, row in tqdm(df_target.iterrows(), total=len(df_target), desc="Verifying Logic"):
    n_p, pres, p_str = evaluate_row(row)
    num_props_list.append(n_p)
    preserved_list.append(pres)
    props_str_list.append(p_str)

df_target["num_propositions"] = num_props_list
df_target["keyword_preserved_in_props"] = preserved_list
df_target["propositions"] = props_str_list

# Xuất file kết quả đã điền đầy đủ và kiểm định chuẩn xác
out_csv = os.path.join(OUT_DIR, f"factoidwiki_full_verified_audit_{MAP_MODE}.csv")
df_target.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"[+] ĐÃ XUẤT THÀNH CÔNG FILE KẾT QUẢ ĐẦY ĐỦ: {out_csv}")

# Thống kê tỷ lệ
valid_df = df_target[df_target["num_propositions"] > 0]
total_valid = len(valid_df)
dropped = valid_df[valid_df["keyword_preserved_in_props"] == False]
dropped_count = len(dropped)
error_rate = (dropped_count / total_valid * 100) if total_valid > 0 else 0

print("="*75)
print("       BÁO CÁO KIỂM ĐỊNH THỰC TẾ TOÀN BỘ DATASET FACTOIDWIKI")
print("="*75)
print(f"Tổng số đoạn văn logic kiểm định thành công : {total_valid:,}")
print(f"Số đoạn văn BỊ CẮT BỎ HOÀN TOÀN ĐIỀU KIỆN : {dropped_count:,} ({error_rate:.1f}%)")
print("\nTop các liên từ bị mô hình Propositionizer làm mất nhiều nhất:")
print(dropped["keyword"].value_counts().head(10).to_string())
print("="*75)


## 6. Trực quan hóa Biểu đồ Báo cáo Khoa học (Publication-Ready Figures)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.set_theme(style="whitegrid")

# 1. Biểu đồ tròn: Tỷ lệ mất điều kiện vs Giữ được
labels = ['Giữ được điều kiện', 'Bị cắt bỏ điều kiện (LỖI LOGIC)']
sizes = [total_valid - dropped_count, dropped_count]
colors = ['#2ecc71', '#e74c3c']
axes[0].pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140, colors=colors, explode=(0, 0.08), textprops={'fontsize': 11, 'weight': 'bold'})
axes[0].set_title(f'Tỷ lệ Bảo tồn Logic của Propositionizer\n(Kiểm định trên {total_valid:,} Đoạn văn)', fontsize=13, weight='bold')

# 2. Biểu đồ cột: Các từ khóa bị cắt cụt nhiều nhất
top_dropped_kw = dropped["keyword"].value_counts().head(8)
sns.barplot(x=top_dropped_kw.values, y=top_dropped_kw.index, ax=axes[1], palette="Reds_r")
axes[1].set_title('Top các Liên từ Logic bị Cắt cụt Nhiều nhất', fontsize=13, weight='bold')
axes[1].set_xlabel('Số lượng ca lỗi (Cases)', fontsize=11)

for i, v in enumerate(top_dropped_kw.values):
    axes[1].text(v + 1, i, f"{v:,}", va="center", fontsize=10, weight="bold")

plt.tight_layout()
chart_file = os.path.join(OUT_DIR, "figure_global_proposition_drop_rate.png")
plt.savefig(chart_file, dpi=300)
plt.show()
print(f"[+] Đã lưu biểu đồ học thuật vào: {chart_file}")


## 7. Tổng kết Báo cáo Khoa học Toàn cục (Academic Deliverable)


In [ ]:
summary_metrics = {
    "Audit Status": "SUCCESSFULLY VERIFIED ON FACTOIDWIKI",
    "Scope Shards": f"{len(shards_to_process)} / 1000 shards",
    "Total Verified Logic Passages": f"{total_valid:,}",
    "Total Condition Truncation Errors": f"{dropped_count:,}",
    "Actual Error Rate": f"{error_rate:.2f}%",
    "Estimated Global Errors across Full Wikipedia": f"{int(152111 * (error_rate/100)):,} passages",
    "Top Vulnerable Connectives": dropped["keyword"].value_counts().head(3).to_dict()
}

summary_json_path = os.path.join(OUT_DIR, "verified_full_audit_summary.json")
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary_metrics, f, indent=2, ensure_ascii=False)

print("[+] BÁO CÁO TỔNG KẾT:")
print(json.dumps(summary_metrics, indent=2, ensure_ascii=False))
